# Setup

In [34]:
# importing libraries 
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    RobustScaler,
    FunctionTransformer
)

from src.data.preprocessing import UNSWPreprocessor

In [2]:
# notebook config
PROJECT_ROOT = Path.cwd().resolve().parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = DATA_DIR / "UNSW_NB15_training-set.csv"
TEST_PATH = DATA_DIR / "UNSW_NB15_testing-set.csv"

print("Project root:", PROJECT_ROOT)
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)

Project root: C:\Projects\Aeges-Q
Train path: C:\Projects\Aeges-Q\data\raw\UNSW_NB15_training-set.csv
Test path: C:\Projects\Aeges-Q\data\raw\UNSW_NB15_testing-set.csv


# Preprocessing Strategy

Based on the exploratory data analysis, the UNSW-NB15 dataset requires
a preprocessing pipeline that addresses categorical encoding, large
differences in feature scale, highly skewed numerical distributions,
and potential feature redundancy.

Rather than applying all transformations universally, multiple
preprocessing variants will be constructed and evaluated according to
their suitability for different downstream models.

In [3]:
# load dataset
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (82332, 45)
Testing shape: (175341, 45)


# Preprocessing Variant A — Base Pipeline

This pipeline establishes the baseline preprocessing strategy for the
UNSW-NB15 dataset. Numerical features are median-imputed, while
categorical features are imputed using the most frequent value and
one-hot encoded.

This representation serves as the reference point against which more
specialized preprocessing strategies will be compared.

In [5]:
# preprocessor 
preprocessor = UNSWPreprocessor()

X_train, y_train, attack_train = (
    preprocessor.split_features_and_target(train_df)
)

X_test, y_test, attack_test = (
    preprocessor.split_features_and_target(test_df)
)

In [15]:
preprocessor.identify_feature_types(X_train)

numerical_features = preprocessor.numerical_features
categorical_features = preprocessor.categorical_features

print(len(numerical_features))
print(len(categorical_features))

print(categorical_features)

39
3
['proto', 'service', 'state']


In [6]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTarget distribution:")
print(y_train.value_counts(normalize=True))

X_train: (82332, 42)
X_test: (175341, 42)

Target distribution:
label
1    0.5506
0    0.4494
Name: proportion, dtype: float64


# Preprocessing Variant B — Standard Scaling

The EDA revealed large differences in numerical feature ranges. While
tree-based models are generally insensitive to feature scaling, scaling
is important for distance-based, margin-based, and quantum machine
learning models.

This variant applies standardization to numerical features while
preserving the existing categorical preprocessing strategy.

In [12]:
# scaled numerical pipeline
scaled_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [13]:
# categorical pipeline 
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [16]:
# scaled preprocessor 
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            scaled_numeric_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [17]:
X_train_scaled = scaled_preprocessor.fit_transform(X_train)

X_test_scaled = scaled_preprocessor.transform(X_test)

In [18]:
print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Scaled train shape: (82332, 190)
Scaled test shape: (175341, 190)


In [19]:
scaled_feature_names = (
    scaled_preprocessor.get_feature_names_out()
)

In [20]:
numeric_feature_indices = [
    i
    for i, feature_name in enumerate(scaled_feature_names)
    if feature_name.startswith("numeric__")
]

In [21]:
X_train_scaled_numeric = (
    X_train_scaled[:, numeric_feature_indices]
)

In [22]:
numeric_means = np.asarray(
    X_train_scaled_numeric.mean(axis=0)
).ravel()

numeric_stds = np.sqrt(
    np.asarray(
        X_train_scaled_numeric.power(2).mean(axis=0)
    ).ravel()
    - numeric_means ** 2
)

In [23]:
print("Maximum absolute mean:", np.abs(numeric_means).max())

print("Minimum std:", numeric_stds.min())
print("Maximum std:", numeric_stds.max())

Maximum absolute mean: 1.5724755515983489e-13
Minimum std: 0.9999999999982245
Maximum std: 1.0000000000011098


# Preprocessing Variant C — Robust Scaling

Several numerical features exhibit heavy skewness and extreme values.
Unlike standard scaling, RobustScaler uses the median and interquartile
range, making it less sensitive to the influence of outliers.

This variant is evaluated as an alternative numerical scaling strategy
while retaining the same categorical preprocessing pipeline.

In [31]:
# numeric pipeline 
robust_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            RobustScaler()
        )
    ]
)

In [30]:
# transformer 
robust_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            robust_numeric_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [26]:
# fit, train, transform and test
X_train_robust = robust_preprocessor.fit_transform(X_train)

X_test_robust = robust_preprocessor.transform(X_test)

print("Robust train shape:", X_train_robust.shape)
print("Robust test shape:", X_test_robust.shape)

Robust train shape: (82332, 190)
Robust test shape: (175341, 190)


In [29]:
# verification
robust_feature_names = robust_preprocessor.get_feature_names_out()

numeric_feature_indices = [
    i
    for i, feature_name in enumerate(robust_feature_names)
    if feature_name.startswith("numeric__")
]

X_train_robust_numeric = X_train_robust[:, numeric_feature_indices]

In [28]:
X_train_robust_numeric_dense = X_train_robust_numeric.toarray()

numeric_medians = np.median(
    X_train_robust_numeric_dense,
    axis=0
)

q1 = np.percentile(
    X_train_robust_numeric_dense,
    25,
    axis=0
)

q3 = np.percentile(
    X_train_robust_numeric_dense,
    75,
    axis=0
)

numeric_iqr = q3 - q1

print("Maximum absolute median:", np.abs(numeric_medians).max())

print("Minimum IQR:", numeric_iqr.min())
print("Maximum IQR:", numeric_iqr.max())

Maximum absolute median: 0.0
Minimum IQR: 0.0
Maximum IQR: 1.0


In [32]:
# finding 0 IQR features 
zero_iqr_features = [
    numerical_features[i]
    for i, iqr in enumerate(numeric_iqr)
    if np.isclose(iqr, 0)
]

zero_iqr_features

['trans_depth',
 'response_body_len',
 'is_ftp_login',
 'ct_ftp_cmd',
 'ct_flw_http_mthd',
 'is_sm_ips_ports']

In [33]:
# non zero IQR features
nonzero_iqr = numeric_iqr[
    ~np.isclose(numeric_iqr, 0)
]

print("Zero-IQR features:", zero_iqr_features)
print("Number of zero-IQR features:", len(zero_iqr_features))

print("\nMinimum non-zero IQR:", nonzero_iqr.min())
print("Maximum non-zero IQR:", nonzero_iqr.max())

Zero-IQR features: ['trans_depth', 'response_body_len', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports']
Number of zero-IQR features: 6

Minimum non-zero IQR: 0.9999999999999999
Maximum non-zero IQR: 1.0


# Preprocessing Variant D - Skew Aware Transformer

# Preprocessing Variant D1 — Broad Log Transformation

The EDA identified several numerical features with heavily right-skewed
distributions. Extreme skewness can cause a small number of large values
to dominate the numerical range, potentially affecting models that are
sensitive to feature distributions and scale.

This variant selectively applies a `log1p` transformation only to
suitable non-negative numerical features with substantial positive
skewness. The transformed features are then standardized, while the
remaining numerical and categorical features follow their respective
preprocessing pipelines.

The goal is to evaluate whether reducing distributional skewness
improves the quality of the feature representation compared with the
base and scaling-only variants.

In [35]:
# candidate features
skewness = X_train[numerical_features].skew()

skewed_features = skewness[
    skewness > 1
].index.tolist()

print("Number of heavily right-skewed features:", len(skewed_features))
print(skewed_features)

Number of heavily right-skewed features: 33
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']


In [36]:
invalid_log_features = [
    feature
    for feature in skewed_features
    if X_train[feature].min() < 0
]

print("Features with negative values:", invalid_log_features)

Features with negative values: []


In [38]:
# log features 
log_features = [
    feature
    for feature in skewed_features
    if feature not in invalid_log_features
]

remaining_numeric_features = [
    feature
    for feature in numerical_features
    if feature not in log_features
]

print("Log-transformed features:", len(log_features))
print("Remaining numerical features:", len(remaining_numeric_features))

Log-transformed features: 33
Remaining numerical features: 6


In [39]:
# log transformed numerical features 
log_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "log_transform",
            FunctionTransformer(np.log1p)
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [40]:
# remaining numerical features
remaining_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [41]:
# skew aware transformer
skew_preprocessor = ColumnTransformer(
    transformers=[
        (
            "log_numeric",
            log_numeric_transformer,
            log_features
        ),
        (
            "remaining_numeric",
            remaining_numeric_transformer,
            remaining_numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [42]:
# fit and transform
X_train_skew = skew_preprocessor.fit_transform(X_train)

X_test_skew = skew_preprocessor.transform(X_test)

print("Skew-aware train shape:", X_train_skew.shape)
print("Skew-aware test shape:", X_test_skew.shape)

Skew-aware train shape: (82332, 190)
Skew-aware test shape: (175341, 190)


In [44]:
# verification : before vs after skewness
# Extract feature names
X_train_log_transformed = X_train_skew[:, :len(log_features)]

X_train_log_dense = X_train_log_transformed.toarray()

skew_comparison = pd.DataFrame({
    "feature": log_features,
    "skew_before": [
        X_train[feature].skew()
        for feature in log_features
    ],
    "skew_after": [
        pd.Series(X_train_log_dense[:, i]).skew()
        for i in range(len(log_features))
    ]
})

skew_comparison["absolute_skew_reduction"] = (
    skew_comparison["skew_before"].abs()
    - skew_comparison["skew_after"].abs()
)

skew_comparison.sort_values(
    by="absolute_skew_reduction",
    ascending=False
)

,feature,skew_before,skew_after,absolute_skew_reduction
19,trans_depth,170.794394,2.989419,1.678050e+02
20,response_body_len,74.635200,4.111906,7.052329e+01
13,djit,60.562275,0.690386,5.987189e+01
3,sbytes,53.778546,1.114123,5.266442e+01
9,dloss,54.465649,1.827054,5.263860e+01
4,dbytes,52.550387,0.248942,5.230144e+01
8,sloss,52.465277,1.265456,5.119982e+01
2,dpkts,49.304127,0.692811,4.861132e+01
1,spkts,47.747777,1.075176,4.667260e+01
11,dinpkt,23.013046,0.749781,2.226327e+01


# Preprocessing Variant D2 — Selective Log Transformation

The broad log transformation experiment showed that `log1p` substantially
reduced skewness for most heavily right-skewed numerical features.
However, several sparse or indicator-like features showed negligible
improvement after transformation.

This variant selectively applies `log1p` only to features for which the
broad transformation demonstrated a meaningful reduction in skewness.
Features with negligible improvement remain untransformed and are only
scaled.

The objective is to determine whether a more targeted transformation
strategy produces a better feature representation than applying the
same transformation to all highly skewed features.

In [45]:
# exclude from log transformation features with skewness reduction less than 0.1
no_log_features = [
    "is_ftp_login",
    "ct_ftp_cmd",
    "is_sm_ips_ports"
]

In [46]:
# selectiive feature groups 
selective_log_features = [
    feature
    for feature in log_features
    if feature not in no_log_features
]

selective_remaining_numeric_features = [
    feature
    for feature in numerical_features
    if feature not in selective_log_features
]

print(
    "Features receiving log transformation:",
    len(selective_log_features)
)

print(
    "Remaining numerical features:",
    len(selective_remaining_numeric_features)
)

print("\nLog-transformed features:")
print(selective_log_features)

print("\nNon-log numerical features:")
print(selective_remaining_numeric_features)

Features receiving log transformation: 30
Remaining numerical features: 9

Log-transformed features:
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst']

Non-log numerical features:
['sttl', 'dttl', 'swin', 'stcpb', 'dtcpb', 'dwin', 'is_ftp_login', 'ct_ftp_cmd', 'is_sm_ips_ports']


In [47]:
# log branch
selective_log_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "log_transform",
            FunctionTransformer(np.log1p)
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [48]:
# remaining numeric branch 
selective_remaining_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [49]:
# transformer 
selective_skew_preprocessor = ColumnTransformer(
    transformers=[
        (
            "log_numeric",
            selective_log_numeric_transformer,
            selective_log_features
        ),
        (
            "remaining_numeric",
            selective_remaining_numeric_transformer,
            selective_remaining_numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [50]:
# fit and transform
X_train_selective_skew = (
    selective_skew_preprocessor.fit_transform(X_train)
)

X_test_selective_skew = (
    selective_skew_preprocessor.transform(X_test)
)

print(
    "Selective skew-aware train shape:",
    X_train_selective_skew.shape
)

print(
    "Selective skew-aware test shape:",
    X_test_selective_skew.shape
)

Selective skew-aware train shape: (82332, 190)
Selective skew-aware test shape: (175341, 190)


# Preprocessing Variant E — Feature Redundancy Reduction

The EDA revealed strong correlations between several numerical features,
indicating potential redundancy in the feature space. Highly correlated
features may provide overlapping information, increasing dimensionality
without necessarily contributing additional predictive value.

This variant investigates correlation-based feature reduction by
systematically identifying highly correlated numerical features and
applying a reproducible pruning strategy.

Rather than arbitrarily removing features, correlated feature groups
will first be identified and analyzed before determining which features
should be retained. The resulting reduced representation will then be
evaluated against the previous preprocessing variants.